# 02 - Feature Engineering
## Create lag features, rolling statistics, seasonality features, and promotion features

In [1]:
import pandas as pd
import numpy as np

DATA_DIR = '../data/raw/'

In [2]:
transactions = pd.read_csv(f'{DATA_DIR}transaction_data.csv')
products = pd.read_csv(f'{DATA_DIR}product.csv')
print(f'Transactions: {len(transactions):,} rows')
print(f'Products: {len(products):,} rows')

Transactions: 2,595,732 rows
Products: 92,353 rows


In [4]:
df = transactions.merge(products[['PRODUCT_ID', 'DEPARTMENT', 'COMMODITY_DESC']], on='PRODUCT_ID', how='left')
df = df.sort_values(['PRODUCT_ID', 'DAY']).reset_index(drop=True)
print(f'Merged dataset: {len(df):,} rows')
df.head()

Merged dataset: 2,595,732 rows


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC,DEPARTMENT,COMMODITY_DESC
0,1228,29046618323,157,25671,1,3.49,3313,0.0,2213,23,0.0,0.0,GROCERY,FRZN ICE
1,358,30707611686,247,25671,1,3.49,3266,0.0,1211,36,0.0,0.0,GROCERY,FRZN ICE
2,325,33046710871,410,25671,4,13.96,3191,0.0,1139,59,0.0,0.0,GROCERY,FRZN ICE
3,1675,30760265177,250,26081,1,0.99,3235,0.0,936,36,0.0,0.0,MISC. TRANS.,NO COMMODITY DESCRIPTION
4,1032,33783848749,458,26093,1,1.59,33904,0.0,2034,66,0.0,0.0,PASTRY,BREAD


In [5]:
# Lag features
for lag in [1, 7, 14]:
    df[f'lag_{lag}'] = df.groupby('PRODUCT_ID')['QUANTITY'].shift(lag).fillna(0)

# Rolling statistics
for window in [7, 14, 28]:
    df[f'rolling_mean_{window}'] = (
        df.groupby('PRODUCT_ID')['QUANTITY']
        .transform(lambda x: x.rolling(window, min_periods=1).mean().shift(1))
        .fillna(0)
    )
    df[f'rolling_std_{window}'] = (
        df.groupby('PRODUCT_ID')['QUANTITY']
        .transform(lambda x: x.rolling(window, min_periods=1).std().shift(1))
        .fillna(0)
    )

print('Lag and rolling features created')

KeyboardInterrupt: 

In [ ]:
# Seasonality / calendar features
df['day_of_week'] = df['DAY'] % 7
df['month'] = ((df['DAY'] / 30).astype(int) % 12) + 1
df['quarter'] = ((df['DAY'] / 30).astype(int) % 12 // 3) + 1
df['week_of_year'] = (df['DAY'] / 7).astype(int) % 52
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

print('Calendar features created')

In [ ]:
# Promotion / discount features
df['retail_disc_abs'] = df['RETAIL_DISC'].abs()
df['coupon_disc_abs'] = df['COUPON_DISC'].abs()
df['total_discount'] = df['retail_disc_abs'] + df['coupon_disc_abs']
df['has_discount'] = (df['total_discount'] > 0).astype(int)
df['discount_rate'] = np.where(
    df['SALES_VALUE'] + df['total_discount'] > 0,
    df['total_discount'] / (df['SALES_VALUE'] + df['total_discount']),
    0
)

print('Discount features created')

In [ ]:
# Product-level aggregate features
product_stats = df.groupby('PRODUCT_ID')['QUANTITY'].agg(['mean', 'std', 'max', 'sum']).fillna(0)
product_stats.columns = ['product_avg_qty', 'product_std_qty', 'product_max_qty', 'product_total_qty']
df = df.merge(product_stats, on='PRODUCT_ID', how='left')

print('Product aggregate features created')

In [ ]:
feature_cols = [
    'DAY', 'PRODUCT_ID', 'DEPARTMENT', 'COMMODITY_DESC',
    'lag_1', 'lag_7', 'lag_14',
    'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28',
    'rolling_std_7', 'rolling_std_14', 'rolling_std_28',
    'day_of_week', 'month', 'quarter', 'week_of_year', 'is_weekend',
    'retail_disc_abs', 'coupon_disc_abs', 'total_discount', 'has_discount', 'discount_rate',
    'product_avg_qty', 'product_std_qty', 'product_max_qty', 'product_total_qty',
    'QUANTITY', 'SALES_VALUE'
]
feature_df = df[feature_cols].copy()

print(f'Feature matrix shape: {feature_df.shape}')
print(f'\nFeature columns: {feature_df.columns.tolist()}')

In [ ]:
# Sample of top 50 products by volume for training
top_products = df.groupby('PRODUCT_ID')['QUANTITY'].sum().nlargest(50).index
sample_df = df[df['PRODUCT_ID'].isin(top_products)]
print(f'Sample for training: {len(sample_df):,} rows')
print(f'Unique products: {sample_df["PRODUCT_ID"].nunique()}')

In [ ]:
feature_df.head()

In [ ]:
# Check for NaN and Inf
print('NaN counts per column:')
print(feature_df.isnull().sum()[feature_df.isnull().sum() > 0])
print('\nFeature matrix ready for modeling')